In [ ]:
import os
import time

from pyspark.sql import SparkSession

os.environ['OBJC_DISABLE_INITIALIZE_FORK_SAFETY'] = 'YES'

try:
    existing_spark = SparkSession.getActiveSession()
    if existing_spark:
        existing_spark.stop()
except:
    pass

for key in list(os.environ.keys()):
    if 'SPARK' in key or 'JAVA_OPTS' in key:
        del os.environ[key]

# --- 2. Cluster Configuration ---
# Format: local-cluster[num_workers, cores_per_worker, memory_per_worker_in_MB]
NUM_EXECUTORS = 2
CORES_PER_EXECUTOR = 6
MEMORY_PER_EXECUTOR_MB = 4096

MASTER_URL = f"local-cluster[{NUM_EXECUTORS}, {CORES_PER_EXECUTOR}, {MEMORY_PER_EXECUTOR_MB}]"

print(f"Running in mode: {MASTER_URL}")

sp_s = (SparkSession.builder
    .master(MASTER_URL)
    .appName("LocalClusterTest")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.executor.cores", "6")
    .config("spark.executor.instances", NUM_EXECUTORS)
    .config("spark.memory.fraction", "0.6")
    .config("spark.sql.shuffle.partitions", "4")  # For tests, less than the default 200
    .getOrCreate()
)

sp_s.sparkContext.setLogLevel("WARN")

# --- 3. Configuration Check ---
print("Session created.")
print(f"Driver Memory Config: {sp_s.conf.get('spark.driver.memory')}")
print(f"Executor Memory Config: {sp_s.conf.get('spark.executor.memory')}")

# Check the number of executors (may take a couple of seconds to start)
time.sleep(3)
num_executors = len(sp_s.sparkContext.parallelize(range(10), NUM_EXECUTORS).glom().collect())
print(f"📊 Active executors (checked via RDD): {num_executors}")

# --- 4. Distribution Test (Example) ---
# To make sure the task went to executors, not stayed on the driver
def print_executor_info(iterator):
    import os
    # Get the executor ID from the process environment variables
    executor_id = os.environ.get('SPARK_EXECUTOR_ID', 'Driver/Local')
    process_id = os.getpid()
    return [f"Executor ID: {executor_id}, PID: {process_id}"]

# Create a dataframe and apply a transformation
df = sp_s.range(0, 10, 1, 4)  # 4 partitions
result = df.rdd.mapPartitions(print_executor_info).collect()

print("\n🖥️ Where tasks were executed:")
for line in result:
    print(line)

sp_s

Running in mode: local-cluster[2, 6, 4096]


26/09/15 16:56:18 WARN Utils: Your hostname, MacBook-Pro-Danil.local resolves to a loopback address: 127.0.0.1; using 10.246.90.43 instead (on interface en0)
26/09/15 16:56:18 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/15 16:56:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Session created.
Driver Memory Config: 4g
Executor Memory Config: 4g


📊 Active executors (checked via RDD): 2

🖥️ Where tasks were executed:
Executor ID: Driver/Local, PID: 13201
Executor ID: Driver/Local, PID: 13190
Executor ID: Driver/Local, PID: 13203
Executor ID: Driver/Local, PID: 13202


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 60790)
Traceback (most recent call last):
  File "/opt/homebrew/Cellar/python@3.11/3.11.14_3/Frameworks/Python.framework/Versions/3.11/lib/python3.11/socketserver.py", line 317, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/opt/homebrew/Cellar/python@3.11/3.11.14_3/Frameworks/Python.framework/Versions/3.11/lib/python3.11/socketserver.py", line 348, in process_request
    self.finish_request(request, client_address)
  File "/opt/homebrew/Cellar/python@3.11/3.11.14_3/Frameworks/Python.framework/Versions/3.11/lib/python3.11/socketserver.py", line 361, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/opt/homebrew/Cellar/python@3.11/3.11.14_3/Frameworks/Python.framework/Versions/3.11/lib/python3.11/socketserver.py", line 755, in __init__
    self.handle()
  File "/Users/danilsamsutdinov/HypEx/.venv/lib/pytho

# AB test 

A/B testing is a research method that allows you to find out people's reaction to any changes. The study shows which of the two versions of the product or offer is better and gives greater effect.

In [4]:
from hypex import HomogeneityTest
from hypex.dataset import Dataset, InfoRole, TargetRole, TreatmentRole
from hypex.utils import BackendsEnum, create_test_data

## Creation of a new test dataset with synthetic data.
It is important to mark the data fields by assigning the appropriate roles:

* FeatureRole: a role for columns that contain features or predictor variables. Our split will be based on them. Applied by default if the role is not specified for the column.
* TreatmentRole: a role for columns that show the treatment or intervention.
* TargetRole: a role for columns that show the target or outcome variable.
* InfoRole: a role for columns that contain information about the data, such as user IDs.

In [6]:
data = Dataset(
    roles={
        "user_id": InfoRole(int),
        "treat": TreatmentRole(),
        "pre_spends": TargetRole(),
        "post_spends": TargetRole(),
        "gender": TargetRole()
    },
    data=create_test_data(),
    backend=BackendsEnum.spark,
    session=sp_s
)
data

,user_id,signup_month,treat,pre_spends,post_spends,age,gender,industry
0,0.0,11.0,1.0,485.0,428.222222,66.0,F,Logistics
1,1.0,0.0,0.0,471.5,414.333333,52.0,F,Logistics
2,2.0,8.0,1.0,468.5,467.555556,26.0,M,Finance
3,3.0,4.0,1.0,486.0,517.444444,50.0,M,Logistics
4,4.0,0.0,0.0,477.5,434.0,26.0,F,Logistics
...,...,...,...,...,...,...,...,...
9995,9995.0,0.0,0.0,464.0,401.444444,61.0,F,Finance
9996,9996.0,4.0,1.0,476.0,517.222222,29.0,F,Finance
9997,9997.0,0.0,0.0,511.0,425.111111,69.0,M,Logistics
9998,9998.0,0.0,0.0,515.0,417.0,69.0,F,Finance


In [7]:
data.roles

{'user_id': Info(<class 'int'>),
 'treat': Treatment(<class 'float'>),
 'pre_spends': Target(<class 'float'>),
 'post_spends': Target(<class 'float'>),
 'gender': Target(<class 'str'>),
 'signup_month': Default(<class 'float'>),
 'age': Default(<class 'float'>),
 'industry': Default(<class 'str'>)}

## Homogeneity Test  

In [8]:
test = HomogeneityTest()
result = test.execute(data)

/Users/danilsamsutdinov/HypEx/hypex/dataset/backends/pandas_backend.py:962: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  new_data = pd.concat([self.data] + [d.data for d in other], axis=axis)


In [9]:
result.resume

,feature,group,control mean,test mean,difference,difference %,TTest pass,TTest p-value,KSTest pass,KSTest p-value,Chi2Test pass,Chi2Test p-value
0,pre_spends,1.0,485.184472,489.908259,4.723786,0.973606,NOT OK,1.222501e-32,NOT OK,1.713950e-17,NaN,NaN
1,pre_spends,nan,485.184472,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,post_spends,1.0,420.081914,482.912971,62.831057,14.956858,NOT OK,0.000000e+00,NOT OK,0.000000e+00,NaN,NaN
3,post_spends,nan,420.081914,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,gender,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NOT OK,2.08818
